# Kontrola enkodera epoki (etap A)
Weryfikacja `dl/encoder.py`: kształty wyjść, liczba parametrów, determinizm w trybie `eval`, channel dropout w trybie `train` oraz pole recepcyjne gałęzi.

In [1]:
import sys
sys.path.append("/home/emil/studia/inzynierka")

import pandas as pd
import torch
from torch.utils.data import DataLoader

from dl.config import (
    BLOCK_STRIDE,
    BRANCH_CONFIG,
    CHANNELS,
    EEG_CHANNEL_INDICES,
    EPOCH_SECONDS,
    SIGNAL_GROUPS,
)
from dl.dataset import PSGEpochDataset
from dl.encoder import PSGEncoder

pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
torch.manual_seed(0)

In [2]:
BATCH_SIZE = 8
model = PSGEncoder()
model

PSGEncoder(
  (branches): ModuleDict(
    (x_200): Branch(
      (stem): Sequential(
        (0): Sequential(
          (0): Conv1d(12, 32, kernel_size=(25,), stride=(2,), padding=(12,), bias=False)
          (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        )
        (1): ReLU(inplace=True)
      )
      (blocks): Sequential(
        (0): ResidualBlock1d(
          (body): Sequential(
            (0): Sequential(
              (0): Conv1d(32, 32, kernel_size=(7,), stride=(2,), padding=(3,), bias=False)
              (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            )
            (1): ReLU(inplace=True)
            (2): Sequential(
              (0): Conv1d(32, 32, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
              (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            )
          )
          (shortcut):

In [3]:
def output_shapes(model, batch):
    embedding, logits = model(batch)
    shapes = {"embedding": tuple(embedding.shape)}
    shapes.update({name: tuple(value.shape) for name, value in logits.items()})
    return pd.Series(shapes, name="shape")

## 1. Losowe tensory

In [4]:
random_batch = {
    group_name: torch.randn(BATCH_SIZE, len(channels), fs * EPOCH_SECONDS)
    for group_name, (fs, channels) in SIGNAL_GROUPS.items()
}
random_batch["ch_mask"] = torch.ones(BATCH_SIZE, len(CHANNELS))
pd.Series({name: tuple(value.shape) for name, value in random_batch.items()}, name="shape")

x_200      (8, 12, 6000)
x_100       (8, 3, 3000)
x_25         (8, 1, 750)
ch_mask          (8, 16)
Name: shape, dtype: object

In [5]:
output_shapes(model.train(), random_batch)

embedding    (8, 128)
stage          (8, 5)
arousal          (8,)
limb             (8,)
resp           (8, 3)
Name: shape, dtype: object

## Wnioski

## 2. Prawdziwy batch (val)

In [6]:
dataset = PSGEpochDataset("val")
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
batch = next(iter(loader))
pd.Series({name: tuple(value.shape) for name, value in batch.items()}, name="shape")

x_200            (8, 12, 6000)
x_100             (8, 3, 3000)
x_25               (8, 1, 750)
ch_mask                (8, 16)
stage                     (8,)
stage_valid               (8,)
arousal                   (8,)
limb                      (8,)
resp                    (8, 3)
arousal_valid             (8,)
resp_valid                (8,)
limb_valid                (8,)
Name: shape, dtype: object

In [7]:
output_shapes(model.train(), batch)

embedding    (8, 128)
stage          (8, 5)
arousal          (8,)
limb             (8,)
resp           (8, 3)
Name: shape, dtype: object

## Wnioski

## 3. Liczba parametrów

In [8]:
def count_parameters(module):
    return sum(parameter.numel() for parameter in module.parameters())

In [9]:
parameters = {f"branch {name}": count_parameters(branch) for name, branch in model.branches.items()}
parameters["projection"] = count_parameters(model.projection)
parameters.update({f"head {name}": count_parameters(head) for name, head in model.heads.items()})
parameters["total"] = count_parameters(model)
pd.Series(parameters, name="parameters")

branch x_200    169376
branch x_100     40368
branch x_25       7928
projection       14464
head stage         645
head arousal       129
head limb          129
head resp          387
total           233426
Name: parameters, dtype: int64

## Wnioski

## 4. Determinizm w trybie eval

In [10]:
model.eval()
with torch.no_grad():
    first = model(batch)
    second = model(batch)

In [11]:
comparison = {"embedding": (first[0], second[0])}
comparison.update({name: (first[1][name], second[1][name]) for name in first[1]})
pd.DataFrame({
    name: {"identical": torch.equal(a, b), "max_abs_diff": (a - b).abs().max().item()}
    for name, (a, b) in comparison.items()
}).T

,identical,max_abs_diff
embedding,True,0.0
stage,True,0.0
arousal,True,0.0
limb,True,0.0
resp,True,0.0


## Wnioski

## 5. Channel dropout w trybie train (p = 1.0)
Przy p = 1.0 każdy obecny kanał jest zerowany, więc po dropoucie powinien zostać dokładnie jeden kanał EEG w każdej próbce, która miała go na wejściu, a pozostałe kanały powinny być wyzerowane. Test powtórzono na wielu batchach, a `ch_mask` porównano przed i po forwardzie.

In [12]:
N_BATCHES = 50
model_full_drop = PSGEncoder(channel_dropout=1.0).train()
eeg_index = torch.tensor(EEG_CHANNEL_INDICES)
non_eeg_index = torch.tensor([i for i in range(len(CHANNELS)) if i not in EEG_CHANNEL_INDICES])

In [13]:
def channel_nonzero(signals):
    return torch.cat([signals[group_name].abs().amax(dim=-1) > 0 for group_name in SIGNAL_GROUPS], dim=1)

In [14]:
rows = []
for _, test_batch in zip(range(N_BATCHES), loader):
    mask_before = test_batch["ch_mask"].clone()
    with torch.no_grad():
        signals = model_full_drop.signals(test_batch)
        model_full_drop(test_batch)
    nonzero = channel_nonzero(signals)
    eeg_in = mask_before[:, eeg_index].bool().any(dim=1)
    eeg_kept = nonzero[:, eeg_index].sum(dim=1)
    rows.append({
        "samples": len(eeg_in),
        "with_eeg_in": int(eeg_in.sum()),
        "with_eeg_kept": int((eeg_kept > 0)[eeg_in].sum()),
        "exactly_one_eeg": int((eeg_kept == 1)[eeg_in].sum()),
        "eeg_kept_without_input": int((eeg_kept > 0)[~eeg_in].sum()),
        "non_eeg_nonzero": int(nonzero[:, non_eeg_index].sum()),
        "ch_mask_unchanged": torch.equal(mask_before, test_batch["ch_mask"]),
    })
dropout_check = pd.DataFrame(rows)

In [15]:
dropout_check.agg({
    "samples": "sum",
    "with_eeg_in": "sum",
    "with_eeg_kept": "sum",
    "exactly_one_eeg": "sum",
    "eeg_kept_without_input": "sum",
    "non_eeg_nonzero": "sum",
    "ch_mask_unchanged": "all",
})

samples                    400
with_eeg_in                397
with_eeg_kept              397
exactly_one_eeg            397
eeg_kept_without_input       0
non_eeg_nonzero              0
ch_mask_unchanged         True
dtype: object

## Wnioski

## 6. Pole recepcyjne gałęzi
Pole recepcyjne ostatniej warstwy konwolucyjnej przed global average pooling, liczone rekurencyjnie: `r += (k - 1) * j`, `j *= s`, gdzie `j` to skumulowany stride. Skrót 1x1 nie zwiększa pola recepcyjnego.

In [16]:
def receptive_field(stem_kernel, stem_stride, block_kernel, channels):
    layers = [(stem_kernel, stem_stride)]
    for _ in range(len(channels) - 1):
        layers += [(block_kernel, BLOCK_STRIDE), (block_kernel, 1)]
    field, jump = 1, 1
    for kernel, stride in layers:
        field += (kernel - 1) * jump
        jump *= stride
    return field, jump

In [17]:
fields = []
for group_name, config in BRANCH_CONFIG.items():
    fs = SIGNAL_GROUPS[group_name][0]
    field, jump = receptive_field(**config)
    fields.append({
        "branch": group_name,
        "fs": fs,
        "epoch_samples": fs * EPOCH_SECONDS,
        "receptive_field_samples": field,
        "receptive_field_seconds": field / fs,
        "total_stride": jump,
        "output_length": model.branches[group_name].blocks(model.branches[group_name].stem(batch[group_name])).shape[-1],
    })
pd.DataFrame(fields).set_index("branch")

,fs,epoch_samples,receptive_field_samples,receptive_field_seconds,total_stride,output_length
branch,,,,,,
x_200,200,6000,565,2.825,32,188
x_100,100,3000,361,3.610,16,188
x_25,25,750,259,10.360,16,47


## Wnioski